# 🧹 Notebook 02 — Data Cleaning

**Project:** Nigeria Disease Surveillance Dashboard  
**Purpose:** Transform raw data into clean, analysis-ready tables.  

**Input:** Raw files from `data/raw/` (produced by Notebook 01)  
**Output:** Clean files to `data/processed/`

**Principle:** Every cleaning decision is documented above the code that  
implements it. Raw data is **never modified** — we always work on copies.

---
**Steps:**
1. Load raw files
2. Standardise state names
3. Clean each disease DataFrame
4. Clean population data
5. Clean rainfall data
6. Merge all diseases into master table
7. Add incidence rates
8. Validate and save

## 1. Environment Setup

In [ ]:
import sys
import warnings
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import Paths, Diseases
from src.utils.logger import get_logger, set_log_level
from src.utils.state_maps import (
    CANONICAL_STATES, CANONICAL_STATE_SET,
    normalise_state_name, STATE_NAME_VARIANTS
)
from src.etl.transform import (
    clean_disease_dataframe,
    clean_population_data,
    clean_rainfall_data,
    merge_all_diseases,
    add_incidence_rate,
)

set_log_level('WARNING')   # Reduce noise during cleaning
logger = get_logger('notebook_02')

print('✅ Environment ready')
print(f'   Looking for raw files in: {Paths.raw}')

## 2. Load Raw Files

Load everything produced by Notebook 01.

In [ ]:
def load_raw(filename: str, filetype: str = 'csv') -> pd.DataFrame:
    """Load a raw file with a clear warning if it doesn't exist."""
    path = Paths.raw / filename
    if not path.exists():
        print(f'  ⚠️  Not found: {path.name} — run Notebook 01 first')
        return pd.DataFrame()
    df = pd.read_csv(path) if filetype == 'csv' else pd.read_excel(path)
    print(f'  ✅ {path.name:<45} {len(df):>6,} rows')
    return df

print('Loading raw files...')
raw_files = {
    Diseases.CHOLERA     : load_raw('ncdc_cholera_raw.csv'),
    Diseases.LASSA       : load_raw('ncdc_lassa_fever_raw.csv'),
    Diseases.MPOX        : load_raw('ncdc_mpox_raw.csv'),
    Diseases.MENINGITIS  : load_raw('ncdc_meningitis_raw.csv'),
    Diseases.YELLOW_FEVER: load_raw('ncdc_yellow_fever_raw.csv'),
}
raw_who        = load_raw('who_raw.csv')
raw_rainfall   = load_raw('rainfall_raw.csv')
raw_facilities = load_raw('health_facilities.csv')
raw_population = load_raw('nigeria_population.xlsx', 'excel')

total = sum(len(df) for df in raw_files.values() if not df.empty)
print(f'\n  Total disease rows loaded: {total:,}')

## 3. State Name Standardisation

**Problem:** Different sources spell state names differently.  
`'FCT'`, `'Abuja'`, `'FCT-Abuja'`, `'Federal Capital Territory'`  
all refer to the same entity.

**Solution:** A canonical mapping that converts all variants to one standard name.  
We inspect the raw state names first to discover any unexpected variants.

In [ ]:
# Discover all unique state name values across all disease files
all_raw_state_names = set()

for disease, df in raw_files.items():
    if df.empty:
        continue
    # Find the state column (may have different names in different files)
    state_col = next(
        (c for c in df.columns if 'state' in c.lower() or 'lga' in c.lower()),
        None
    )
    if state_col:
        vals = df[state_col].dropna().unique()
        all_raw_state_names.update(str(v).strip() for v in vals)

print(f'Unique raw state name values found: {len(all_raw_state_names)}')
print('\nMapping each to canonical name:')

mapping_results = {'canonical': [], 'national': [], 'unknown': []}
for raw in sorted(all_raw_state_names):
    canonical = normalise_state_name(raw)
    if canonical in CANONICAL_STATE_SET:
        mapping_results['canonical'].append((raw, canonical))
    elif canonical == 'NATIONAL':
        mapping_results['national'].append(raw)
    else:
        mapping_results['unknown'].append((raw, canonical))

print(f'  ✅  Canonical : {len(mapping_results["canonical"])} values')
print(f'  🗑️   National  : {len(mapping_results["national"])} values (will be dropped)')
print(f'  ❓  Unknown   : {len(mapping_results["unknown"])} values (investigate below)')

if mapping_results['unknown']:
    print('\n  ⚠️  Unknown state names — add to STATE_NAME_VARIANTS if legitimate:')
    for raw, mapped in mapping_results['unknown'][:15]:
        print(f'    "{raw}" → {mapped}')

## 4. Clean Each Disease

The `clean_disease_dataframe()` function applies a consistent  
cleaning pipeline to each disease's raw DataFrame:

1. Drop completely empty rows
2. Standardise column names
3. Normalise state names → canonical form
4. Drop NATIONAL aggregate rows
5. Parse numeric columns (remove commas, handle dashes)
6. Add disease name column
7. Parse epi-week + year → `report_date`
8. Calculate CFR from raw counts
9. Assign data quality flags

In [ ]:
set_log_level('INFO')   # Show cleaning progress

cleaned = {}
print('Cleaning disease DataFrames...\n')

for disease, raw_df in raw_files.items():
    if raw_df.empty:
        print(f'  ⏭️  Skipping {disease} — no raw data')
        continue

    clean_df = clean_disease_dataframe(raw_df, disease)

    if clean_df.empty:
        print(f'  ⚠️  {disease} produced no clean rows')
    else:
        cleaned[disease] = clean_df

print(f'\n✅ {len(cleaned)}/{len(raw_files)} diseases cleaned successfully')

### 4.1 Inspect cleaning results

For each disease, examine what came out and verify the quality flags.

In [ ]:
for disease, df in cleaned.items():
    print(f'\n{"─"*50}')
    print(f'  {disease}')
    print(f'  Rows        : {len(df):,}')
    print(f'  States      : {df["state"].nunique()}/37')
    if 'report_date' in df.columns:
        valid_dates = df['report_date'].dropna()
        if not valid_dates.empty:
            print(f'  Date range  : {valid_dates.min().date()} → {valid_dates.max().date()}')
    print(f'  Quality flags:')
    for flag, count in df['data_quality_flag'].value_counts().items():
        pct = count / len(df) * 100
        print(f'    {flag:<35} {count:>5,} ({pct:.1f}%)')

    # Check: are any states missing?
    present = set(df['state'].unique())
    missing = sorted(set(CANONICAL_STATES) - present)
    if missing:
        print(f'  Missing states ({len(missing)}): {", ".join(missing[:5])}')
        if len(missing) > 5:
            print(f'    ... and {len(missing)-5} more')

## 5. Clean Population Data

In [ ]:
print('Cleaning population data...')
clean_pop = clean_population_data(raw_population)

if clean_pop.empty:
    print('⚠️  Population data not available — incidence rates will be null')
else:
    print(f'✅ Population data: {len(clean_pop)} states')
    print(f'\n  Population range:')
    print(f'    Min: {clean_pop["population"].min():>12,} ({clean_pop.loc[clean_pop["population"].idxmin(), "state"]})')
    print(f'    Max: {clean_pop["population"].max():>12,} ({clean_pop.loc[clean_pop["population"].idxmax(), "state"]})')
    print(f'    Sum: {clean_pop["population"].sum():>12,} (total Nigeria)')
    print()
    display(clean_pop.set_index('state')['population']
            .sort_values(ascending=False).head(10)
            .rename('Population').to_frame())

## 6. Clean Rainfall Data

In [ ]:
print('Cleaning rainfall data...')
clean_rain = clean_rainfall_data(raw_rainfall)

if clean_rain.empty:
    print('⚠️  Rainfall data not available')
else:
    print(f'✅ Rainfall data: {len(clean_rain):,} records')
    print(f'   States  : {clean_rain["state"].nunique()}')
    print(f'   Years   : {clean_rain["year"].min()}–{clean_rain["year"].max()}')
    print(f'   Nulls   : {clean_rain["rainfall_mm"].isna().sum()}')

    # Quick seasonality check — mean rainfall by month across all states
    monthly_avg = clean_rain.groupby('month')['rainfall_mm'].mean()
    fig, ax = plt.subplots(figsize=(10, 3))
    monthly_avg.plot(kind='bar', ax=ax, color='#185FA5', alpha=0.8)
    ax.set_title('Mean Monthly Rainfall (All States, All Years)', fontsize=12)
    ax.set_xlabel('Month')
    ax.set_ylabel('Rainfall (mm)')
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                         'Jul','Aug','Sep','Oct','Nov','Dec'],
                        rotation=0)
    plt.tight_layout()
    plt.show()
    print('\n  Rainy season (Apr–Oct) clearly visible ✅')

## 7. Merge All Diseases + Add Incidence Rates

Combine all cleaned disease DataFrames into one master table,  
then join population data to compute incidence per 100,000.

In [ ]:
print('Merging all diseases into master table...')
master_df = merge_all_diseases(cleaned)

if master_df.empty:
    print('❌ Merge produced empty result — check cleaning steps above')
else:
    print(f'✅ Master table: {len(master_df):,} rows')

    print('\nAdding incidence rates...')
    master_df = add_incidence_rate(master_df, clean_pop)

    print(f'\nMaster DataFrame summary:')
    print(f'  Rows        : {len(master_df):,}')
    print(f'  Diseases    : {master_df["disease"].nunique()} ({list(master_df["disease"].unique())})')
    print(f'  States      : {master_df["state"].nunique()}/37')
    if 'report_date' in master_df.columns:
        valid = master_df['report_date'].dropna()
        if not valid.empty:
            print(f'  Date range  : {valid.min().date()} → {valid.max().date()}')
    print(f'  Has incidence: {master_df["incidence_per_100k"].notna().sum():,} rows ({master_df["incidence_per_100k"].notna().mean()*100:.1f}%)')
    print(f'\n  Quality flags:')
    for flag, count in master_df['data_quality_flag'].value_counts().items():
        pct = count/len(master_df)*100
        print(f'    {flag:<35} {count:>6,} ({pct:.1f}%)')

## 8. Cross-validation Check

Compare total national case counts from NCDC vs WHO where available.

In [ ]:
# Cross-validate NCDC totals against WHO data (where comparable)
print('Cross-validation: NCDC vs WHO totals\n')

ncdc_totals = (
    master_df.groupby('disease')['confirmed_cases']
    .sum()
    .rename('NCDC_total')
)
print('NCDC national totals by disease:')
for disease, total in ncdc_totals.items():
    print(f'  {disease:<20} {total:>8,}')

print('\n✅ If WHO totals are available, compare here.')
print('   Discrepancies > 20% warrant investigation.')

## 9. Save Processed Files

In [ ]:
print('Saving cleaned files to data/processed/...')
Paths.processed.mkdir(parents=True, exist_ok=True)

# Master surveillance file
master_path = Paths.processed / 'master_surveillance.csv'
master_df.to_csv(master_path, index=False)
print(f'  ✅ master_surveillance.csv  ({master_path.stat().st_size/1024:.1f} KB)')

# Individual disease files
for disease in master_df['disease'].unique():
    sub      = master_df[master_df['disease'] == disease]
    fname    = disease.lower().replace(' ', '_')
    sub_path = Paths.processed / f'{fname}_clean.csv'
    sub.to_csv(sub_path, index=False)
    print(f'  ✅ {sub_path.name:<40} ({len(sub):,} rows)')

# Population file
if not clean_pop.empty:
    pop_path = Paths.processed / 'population_clean.csv'
    clean_pop.to_csv(pop_path, index=False)
    print(f'  ✅ population_clean.csv')

# Rainfall file
if not clean_rain.empty:
    rain_path = Paths.processed / 'rainfall_clean.csv'
    clean_rain.to_csv(rain_path, index=False)
    print(f'  ✅ rainfall_clean.csv')

## 10. Summary

In [ ]:
print('='*55)
print('  NOTEBOOK 02 — CLEANING SUMMARY')
print('='*55)
print(f'  Timestamp    : {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print()
if not master_df.empty:
    print(f'  Master rows  : {len(master_df):,}')
    print(f'  Diseases     : {master_df["disease"].nunique()}')
    print(f'  States       : {master_df["state"].nunique()}/37')
    total_clean = (master_df['data_quality_flag']=='CLEAN').sum()
    pct_clean   = total_clean/len(master_df)*100
    print(f'  Clean rows   : {total_clean:,} ({pct_clean:.1f}%)')
print()
print('  Processed files saved:')
for f in sorted(Paths.processed.glob('*.csv')):
    print(f'    ✅ {f.name}')
print()
print('  ➡️  Next step: Run Notebook 03 — EDA')
print('='*55)